In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
# 1.1 Globla Configuration: Define Fleet and Mission 
    # Reference: https://www.eviation.com/wp-content/uploads/2022/09/Eviation-First-Flight-Press-Release-9.27.22.docx-1.pdf

SIMULATION_CONFIG = {
    # Official stats
    'battery_kwh': 850,
    'max_payload_lbs': 2500,
    # Assumptions
    'min_payload_lbs': 500,     
    'model_name': 'Alice_V1',
    'ambient_temp_range': (-5, 45),
    'firmware_options': ['v2.1', 'v2.2'],
    'freight_rate_per_lb': 2.5  
}

In [3]:
# 1.2 Build assets - create 10 unique aircrafts 
# Table 1: dim_aircraft
    #Assumptions:
        # 1. All aircraft are identical model （Alice_V1) with the same 850 kWh capacity.
        # 2. All aircraft have 100% SoH set to 100%
        # 3. Ambient temperature varies per mission (-5°C to 45°C), impacting battery discharge rate.
        # 4. Payloads are randomized (500-2500 lbs) per sortie, representing realistic regional freight demand.
        # 5. A/B Testing: Firmware v2.1 and v2.2 are tested across the fleet to evaluate energy efficiency optimization.

aircraft_ids = [f'N{i:03d}AL' for i in range(101, 111)]
dim_aircraft = pd.DataFrame({
    'tail_number': aircraft_ids,
    'model': SIMULATION_CONFIG['model_name'],
    'battery_capacity_kwh': SIMULATION_CONFIG['battery_kwh'],
    'max_payload_lbs': SIMULATION_CONFIG['max_payload_lbs'],
    'firmware_version': np.random.choice(SIMULATION_CONFIG['firmware_options'], len(aircraft_ids))
})

print(dim_aircraft.shape)
dim_aircraft.head()

(10, 5)


,tail_number,model,battery_capacity_kwh,max_payload_lbs,firmware_version
0,N101AL,Alice_V1,850,2500,v2.1
1,N102AL,Alice_V1,850,2500,v2.2
2,N103AL,Alice_V1,850,2500,v2.2
3,N104AL,Alice_V1,850,2500,v2.2
4,N105AL,Alice_V1,850,2500,v2.2


In [4]:
# Table 1: dim_aircraft
dim_aircraft.to_csv('aviation_data/dim_aircraft.csv', index=False)

In [5]:
# 1.3 Create 100 random flight missions
# Table 2: flight_summary.csv
    # Generate a stochastic mission registry
    # Payload and Ambient Temp vary to evaluate fleet performance across different operational conditions.
        #   payload_lbs refers to Useful Payload
        #   Gross Revenue per Sortie, calculated based on a regional express freight rate at ($2.5/lb).
        #   Outside Air Temperature is assigned at the flight level and assumed constant.

num_flights = 100
np.random.seed(42) 

rate_per_lb = SIMULATION_CONFIG['freight_rate_per_lb']

payloads = np.random.uniform(SIMULATION_CONFIG['min_payload_lbs'], SIMULATION_CONFIG['max_payload_lbs'], num_flights)
temps = np.random.uniform(SIMULATION_CONFIG['ambient_temp_range'][0], SIMULATION_CONFIG['ambient_temp_range'][1], num_flights)

flight_summary = pd.DataFrame({
    'flight_id': [f'FL-{i:05d}' for i in range(1, num_flights + 1)],
    'tail_number': np.random.choice(dim_aircraft['tail_number'], num_flights),
    'payload_lbs': payloads,
    'ambient_temp_c': temps,
    'freight_rate_per_lb': rate_per_lb,
    'ticket_revenue': payloads * rate_per_lb
})

print(flight_summary.shape)
flight_summary.head()

(100, 6)


,flight_id,tail_number,payload_lbs,ambient_temp_c,freight_rate_per_lb,ticket_revenue
0,FL-00001,N108AL,1249.080238,-3.428541,2.5,3122.700594
1,FL-00002,N104AL,2401.428613,26.820521,2.5,6003.571532
2,FL-00003,N101AL,1963.987884,10.717799,2.5,4909.969709
3,FL-00004,N108AL,1697.316968,20.428535,2.5,4243.292421
4,FL-00005,N104AL,812.037281,40.378324,2.5,2030.093202


In [6]:
# Table 2: flight_summary.csv
os.makedirs('aviation_data', exist_ok=True)
flight_summary.to_csv('aviation_data/flight_summary.csv', index=False)

In [10]:
# 1.3 Digital Twin: 
# To simulate battery SOC depletion based on mission phases, payload, and temperature.
# Note: We use 1Hz granularity (1 data point per second) to capture flight dynamics.
# Table 3: Fact_Telemetry_High_Freq 
    #Assumption：
        # Physics Factors
        # Consolidate Phases: 
            # CLIMB: Includes Take-off and climb.  -- peak thermal and electrical stress period (1400kW scale).
            # CRUISE: Steady-state at 15,000 ft. -- longest duration
            # LANDING: Descent and landing, power-down phase. -- minimal used

PHYSICS_ASSUMPTIONS = {
    'payload_drain_coeff': 0.005,   # 0.5% energy drain per 1000 lbs of payload
    'thermal_drain_coeff': 0.0001,      # 0.01% drain per °C deviation from 25°C
    'ideal_temp': 25,         # Target temp for battery chemistry
    'faa_reserve': 0.20,       # 20% FAA VFR safety buffer
    'climb_load': 0.025,
    'cruise_load': 0.018,
    'land_load': 0.008,
}

def simulate_mission(flight_record, config, physics):
    f_id = flight_record['flight_id']
    payload = flight_record['payload_lbs']
    temp = flight_record['ambient_temp_c']
    
    # Pre-calculate stressors (Vectorized inputs for the loop)
    payload_factor = (payload / config['max_payload_lbs']) * physics['payload_drain_coeff']
    temp_factor = abs(temp - physics['ideal_temp']) * physics['thermal_drain_coeff']
    
    current_soc = 100.0
    telemetry = []
    
     # 75-minute simulation window (4500 seconds)
    for second in range(4500):
        # Phase Modeling
        # 1. CLIMB
        if second < 450:     
            phase, phase_load = 'CLIMB', physics['climb_load']
            altitude = second * 33.3  # Climb to 15,000 ft
        # 2. CRUISE
        elif second < 3800:    
            phase, phase_load = 'CRUISE', physics['cruise_load']
            altitude = 15000
        # 3. LANDING
        else:              
            phase, phase_load = 'LANDING', physics['land_load']
            altitude = max(0, 15000 - ((second - 3800) * 22))
            
        # Physics Engine Calculation
        drain = phase_load + payload_factor + temp_factor
        current_soc -= drain
        
        # Set SOC Boundary 
        if current_soc <= 0:
            current_soc = 0
            
        telemetry.append({
            'flight_id': f_id,
            'timestamp_sec': second,
            'altitude_ft': round(altitude, 1),
            'battery_soc_pct': round(current_soc, 2),
            'phase': phase
        })
        
        # Early Stopping Logic
        if current_soc == 0:
            break
            
    return telemetry

fleet_telemetry = []

for _, flight in flight_summary.iterrows():
    telemetry_data = simulate_mission(flight, SIMULATION_CONFIG, PHYSICS_ASSUMPTIONS)
    fleet_telemetry.extend(telemetry_data)

fact_telemetry_high_freq = pd.DataFrame(fleet_telemetry)

print({len(fact_telemetry_high_freq)})
fact_telemetry_high_freq.head(10)

{442297}


,flight_id,timestamp_sec,altitude_ft,battery_soc_pct,phase
0,FL-00001,0,0.0,99.97,CLIMB
1,FL-00001,1,33.3,99.94,CLIMB
2,FL-00001,2,66.6,99.91,CLIMB
3,FL-00001,3,99.9,99.88,CLIMB
4,FL-00001,4,133.2,99.85,CLIMB
5,FL-00001,5,166.5,99.82,CLIMB
6,FL-00001,6,199.8,99.79,CLIMB
7,FL-00001,7,233.1,99.76,CLIMB
8,FL-00001,8,266.4,99.73,CLIMB
9,FL-00001,9,299.7,99.70,CLIMB


In [9]:
# Table 3: Fact_Telemetry_High_Freq 
fact_telemetry_high_freq.to_csv('aviation_data/fact_telemetry_high_freq.csv', index=False)